# mcp

> the vault as MCP tools any client can drive

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

The same `cmd` the CLI uses: a `Vault` method becomes a tool whose schema and description are the
method's own signature and docstring. An agent that can write notes back into the same vault it
reads from accumulates rather than restarts.

In [ ]:
#| export
import json, sys
from functools import wraps
from vishalakshi.cli import cmd, jsonable, vault

try: from mcp.server.fastmcp import FastMCP
except ImportError as e:
    raise ImportError("vishalakshi-mcp needs the `mcp` package — install with "
                      "`pip install 'vishalakshi[mcp]'`") from e

In [ ]:
#| export
TOOLS = ('stats find sections context ask read related toc map sources grab url web arxiv youtube '
         'add_file add_dir note connect forget apis harvest watch watches poll unwatch '
         'index_code code_search symbol where_to_add grep federate').split()

mcp = FastMCP('vishalakshi', instructions=(
    'A personal research vault: web pages, papers, transcripts, files, code and notes in one '
    'searchable corpus. `context` is the main tool — it returns whole sections plus what they '
    'connect to, which is what you want before answering a question. `find` locates things, '
    '`read` pulls one section in full, `related` answers "what else reads like this". `grab` '
    'files anything you point it at; `note` writes your own conclusions back so they are searched '
    'alongside the sources. Run `connect` after a batch of adds to enable the associative leg.'))

def as_tool(name:str):
    'Register one `Vault` method as an MCP tool, forcing its result through JSON.'
    f = cmd(name)
    g = wraps(f)(lambda **kw: json.loads(json.dumps(f(**kw), default=jsonable)))
    g.__signature__, g.__delwrap__, g.__annotations__ = f.__signature__, f.__delwrap__, f.__annotations__
    return mcp.tool(name=name)(g)

for _t in TOOLS: as_tool(_t)

def main():
    'Entry point for `vishalakshi-mcp`. stdio by default; `--http` for Streamable HTTP.'
    mcp.run(transport='streamable-http' if '--http' in sys.argv[1:] else 'stdio')

## Try it

In [ ]:
tools = {t.name: t for t in await mcp.list_tools()}
len(tools), tools['context'].description[:120]

In [ ]:
test_eq(sorted(tools), sorted(TOOLS))
test_eq(tools['find'].inputSchema['properties']['limit']['default'], 10)
test_eq(tools['find'].inputSchema['required'], ['q'])